In [37]:
# ============================================================
#  RAID-SecOps ML Pipeline — Reconstructed
#  Dataset  : UNSW-NB15
#  Models   : Isolation Forest + Random Forest (Hybrid)
#  Output   : Trained model artifacts + role-aware alerts
# ============================================================

# ─────────────────────────────────────────────────────────────
# STAGE 0 — IMPORTS


import os
import json
import warnings
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base             import clone
from sklearn.compose          import ColumnTransformer
from sklearn.ensemble         import IsolationForest, RandomForestClassifier
from sklearn.impute           import SimpleImputer
from sklearn.inspection       import permutation_importance
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
)
from sklearn.model_selection  import (
    train_test_split, RandomizedSearchCV,
    StratifiedKFold, cross_val_score
)
from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("[OK] Stage 0 — Imports complete")

[OK] Stage 0 — Imports complete


In [38]:
# ─────────────────────────────────────────────────────────────
# STAGE 1 — CONFIGURATION
#
# Why: Keep all settings in one place.
# If you move the project to another machine, only edit here.
#
# KEY CHANGE FROM ORIGINAL:
# - Paths now use Path() which works on Windows AND Linux/Mac
# - OUTPUT_DIR is relative to this script, not hardcoded
# - Added SAVE_ARTIFACTS flag so you can turn model saving on/off
# ─────────────────────────────────────────────────────────────

# ── Find where this script lives ─────────────────────────────
# __file__ gives us the path of this script itself.
# .parent gives the folder it's in.
# This means the paths work on ANY machine, not just yours.
SCRIPT_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()

# ── Dataset paths ─────────────────────────────────────────────
# Place your UNSW-NB15 CSV files in the same folder as this script.
# Or change these paths to wherever your files are.
TRAIN_PATH = SCRIPT_DIR / "UNSW_NB15_training-set.csv"
TEST_PATH  = SCRIPT_DIR / "UNSW_NB15_testing-set.csv"

# ── Output folder ─────────────────────────────────────────────
# All plots, metrics JSON, and saved models go here.
OUTPUT_DIR = SCRIPT_DIR / "raid_outputs"
MODEL_DIR  = SCRIPT_DIR / "raid_outputs" / "models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

# ── Random Forest settings ────────────────────────────────────
RF_BASE_PARAMS = {
    "n_estimators":  300,
    "random_state":  RANDOM_STATE,
    "n_jobs":        -1,
    "class_weight":  "balanced",   # handles UNSW-NB15 class imbalance
}

# ── Isolation Forest settings ─────────────────────────────────
IF_BASE_PARAMS = {
    "n_estimators": 300,
    "random_state": RANDOM_STATE,
    "n_jobs":       -1,
    "contamination": "auto",       # auto-estimates contamination rate
}

# ── Hyperparameter tuning ─────────────────────────────────────
# Set to True to run RandomizedSearchCV (slower but better results).
# Set to False to train quickly with default params.
RUN_RF_TUNING = True
CV_FOLDS      = 5

# ── Artifact saving ───────────────────────────────────────────
# Set to True to save trained models as .pkl files.
# These .pkl files are what the FastAPI inference endpoint loads.
SAVE_ARTIFACTS = True

print("[OK] Stage 1 — Configuration complete")
print(f"     Train path : {TRAIN_PATH}")
print(f"     Test path  : {TEST_PATH}")
print(f"     Output dir : {OUTPUT_DIR}")
print(f"     Model dir  : {MODEL_DIR}")


[OK] Stage 1 — Configuration complete
     Train path : C:\Users\kelvi\Desktop\MS CYBERSECURITY\Spring 2026\Capstone\RAID SecOps\raid-secops\ML\UNSW_NB15_training-set.csv
     Test path  : C:\Users\kelvi\Desktop\MS CYBERSECURITY\Spring 2026\Capstone\RAID SecOps\raid-secops\ML\UNSW_NB15_testing-set.csv
     Output dir : C:\Users\kelvi\Desktop\MS CYBERSECURITY\Spring 2026\Capstone\RAID SecOps\raid-secops\ML\raid_outputs
     Model dir  : C:\Users\kelvi\Desktop\MS CYBERSECURITY\Spring 2026\Capstone\RAID SecOps\raid-secops\ML\raid_outputs\models


In [71]:
# ─────────────────────────────────────────────────────────────
# STAGE 2 — MITRE ATT&CK MAPPING  (FIXED)
#
# FIX: UNSW-NB15 uses "Backdoor" (no s) not "Backdoors"
# We now use the EXACT category names from the dataset.
# Run: train_df["attack_cat"].unique() to see all names.
#
# UNSW-NB15 exact category names:
#   Normal, Generic, Exploits, Fuzzers, DoS,
#   Reconnaissance, Analysis, Backdoor, Shellcode, Worms
# ─────────────────────────────────────────────────────────────

MITRE_MAP = {
    "Fuzzers":        "T1190 – Exploit Public-Facing Application",
    "Analysis":       "T1046 – Network Service Scanning",
    "Backdoor":       "T1543 – Create or Modify System Process",   # FIX: was "Backdoors"
    "Backdoors":      "T1543 – Create or Modify System Process",   # keep both for safety
    "DoS":            "T1498 – Network Denial of Service",
    "Exploits":       "T1203 – Exploitation for Client Execution",
    "Generic":        "T1078 – Valid Accounts",
    "Reconnaissance": "T1595 – Active Scanning",
    "Shellcode":      "T1059 – Command and Scripting Interpreter",
    "Worms":          "T1210 – Exploitation of Remote Services",
    "Normal":         "—",
    "Unknown":        "—",
}

BUSINESS_IMPACT_MAP = {
    "Fuzzers":        "Application exploitation attempt — risk of service disruption or data exposure",
    "Analysis":       "Network reconnaissance — attacker may be mapping infrastructure prior to attack",
    "Backdoor":       "Persistent access established — attacker may maintain long-term presence",   # FIX
    "Backdoors":      "Persistent access established — attacker may maintain long-term presence",
    "DoS":            "Service availability risk — operations, SLAs, and revenue may be impacted",
    "Exploits":       "Client-side exploitation — endpoint compromise likely if unmitigated",
    "Generic":        "Credential abuse — account takeover risk, potential data breach",
    "Reconnaissance": "Pre-attack intelligence gathering — indicates targeted attack in progress",
    "Shellcode":      "Code execution attempt — malware or ransomware deployment likely next step",
    "Worms":          "Lateral movement risk — threat may self-propagate across the network",
    "Normal":         "No threat detected — routine network behaviour",
    "Unknown":        "Classification incomplete — manual review required",
}

REGULATORY_MAP = {
    "Fuzzers":        "GDPR Art.32 (technical security measures) — review encryption at rest",
    "Analysis":       "No immediate notification — log for trend analysis",
    "Backdoor":       "GDPR Art.33 — potential breach, 72-hour notification window starts",   # FIX
    "Backdoors":      "GDPR Art.33 — potential breach, 72-hour notification window starts",
    "DoS":            "SLA obligations — notify customers if service disruption exceeds thresholds",
    "Exploits":       "GDPR Art.33 if data accessed — assess scope before notification",
    "Generic":        "GDPR Art.33 if credentials compromised — assess account exposure",
    "Reconnaissance": "No immediate obligation — monitor for follow-on attack",
    "Shellcode":      "GDPR Art.33 likely — code execution implies potential data access",
    "Worms":          "GDPR Art.33 — lateral movement risk, broad scope assessment required",
    "Normal":         "No regulatory action required",
    "Unknown":        "Pending classification — reassess when attack type confirmed",
}

ANALYST_INVESTIGATION_MAP = {
    "Fuzzers": [
        "Check web application logs for malformed input patterns",
        "Review WAF alerts for parameter fuzzing signatures",
        "Inspect source IP for prior reconnaissance activity",
        "Correlate with T1190 — public-facing app exploitation attempts",
    ],
    "Analysis": [
        "Review port scan logs — look for SYN flood or stealth scan patterns",
        "Check if source IP has prior connection history in SIEM",
        "Correlate with T1046 — network service discovery attempts",
        "Review firewall deny logs for unusual destination port ranges",
    ],
    "Backdoor": [   # FIX: was "Backdoors"
        "Inspect process creation logs — look for new services or scheduled tasks",
        "Check registry Run keys and startup folder for persistence",
        "Correlate with T1543 — new system process creation events",
        "Review outbound C2 traffic — check for unexpected external connections",
    ],
    "Backdoors": [  # keep both for safety
        "Inspect process creation logs — look for new services or scheduled tasks",
        "Check registry Run keys and startup folder for persistence",
        "Correlate with T1543 — new system process creation events",
        "Review outbound C2 traffic — check for unexpected external connections",
    ],
    "DoS": [
        "Monitor bandwidth utilisation — compare to baseline",
        "Identify flood source IPs — check if spoofed",
        "Correlate with T1498 — network denial of service patterns",
        "Check if traffic targets a single port or service",
    ],
    "Exploits": [
        "Review client endpoint logs for unusual process spawning",
        "Check for Office macro execution or PDF reader processes",
        "Correlate with T1203 — client execution exploitation",
        "Inspect email gateway logs for malicious attachments",
    ],
    "Generic": [
        "Check authentication logs for credential stuffing patterns",
        "Review VPN and remote access logs for unusual logins",
        "Correlate with T1078 — valid account abuse",
        "Inspect privileged account activity for abnormal access",
    ],
    "Reconnaissance": [
        "Identify if scan is targeted (specific hosts) or broad (subnet sweep)",
        "Check if source IP is internal — may indicate compromised host",
        "Correlate with T1595 — active scanning behaviour",
        "Review DNS query logs for domain enumeration patterns",
    ],
    "Shellcode": [
        "Isolate affected host immediately — code execution confirmed",
        "Capture memory dump before any remediation",
        "Correlate with T1059 — command interpreter invocation",
        "Review PowerShell, cmd, and script block logs",
    ],
    "Worms": [
        "Identify initial infected host and patient zero",
        "Map propagation path — which hosts were contacted",
        "Correlate with T1210 — remote service exploitation",
        "Check SMB and RDP logs for lateral movement indicators",
    ],
}

ENGINEER_REMEDIATION_MAP = {
    "Fuzzers": [
        "Deploy WAF rule to block malformed input from source IP",
        "Enable input validation logging on affected application tier",
        "Review and patch public-facing application to latest version",
        "Update IDS signature set for fuzzing patterns",
    ],
    "Analysis": [
        "Block source IP at perimeter firewall — add to threat intelligence feed",
        "Enable port scan detection rule in SIEM",
        "Review firewall rule set for unnecessarily open ports",
        "Deploy honeypot on commonly scanned ports to detect follow-on activity",
    ],
    "Backdoor": [   # FIX: was "Backdoors"
        "Terminate suspicious processes immediately",
        "Remove unauthorised services or scheduled tasks",
        "Re-image affected host if persistence mechanism confirmed",
        "Deploy endpoint detection rule for new service installation events",
    ],
    "Backdoors": [
        "Terminate suspicious processes immediately",
        "Remove unauthorised services or scheduled tasks",
        "Re-image affected host if persistence mechanism confirmed",
        "Deploy endpoint detection rule for new service installation events",
    ],
    "DoS": [
        "Activate rate limiting on affected service",
        "Engage upstream ISP for traffic scrubbing if volumetric attack",
        "Deploy BGP blackhole routing for identified flood sources",
        "Enable connection throttling at load balancer",
    ],
    "Exploits": [
        "Patch or isolate vulnerable application immediately",
        "Enable exploit mitigation — DEP, ASLR, Control Flow Guard",
        "Deploy application allow-listing on affected endpoint",
        "Update anti-exploit signatures in EDR solution",
    ],
    "Generic": [
        "Force password reset on all affected accounts",
        "Enable MFA if not already active on compromised accounts",
        "Revoke active sessions for affected users",
        "Update account lockout policy to limit brute force attempts",
    ],
    "Reconnaissance": [
        "Block source IP and add to threat intelligence feed",
        "Enable network deception — redirect scans to honeypot",
        "Review externally exposed services — disable unused ones",
        "Update SIEM correlation rule for scan-then-exploit sequences",
    ],
    "Shellcode": [
        "Immediately isolate host from network",
        "Collect forensic evidence before any changes",
        "Deploy memory scanning across all endpoints in same segment",
        "Block outbound C2 connections at perimeter — check threat intel for known IPs",
    ],
    "Worms": [
        "Network segment infected hosts immediately — update VLAN rules",
        "Block SMB and RDP between segments if not required",
        "Deploy patch for exploited vulnerability across all endpoints",
        "Run full AV scan with latest signatures on all reachable hosts",
    ],
}

print("[OK] Stage 2 FIXED — MITRE map uses exact UNSW-NB15 category names")
print("     Both 'Backdoor' and 'Backdoors' now mapped correctly")


[OK] Stage 2 FIXED — MITRE map uses exact UNSW-NB15 category names
     Both 'Backdoor' and 'Backdoors' now mapped correctly


In [40]:
#  ─────────────────────────────────────────────────────────────
# STAGE 3 — HELPER FUNCTIONS
#
# Why: Small utility functions used in many places.
# Keeping them here avoids repeating code everywhere.
# ─────────────────────────────────────────────────────────────

def print_header(title: str):
    """Print a clean section divider."""
    print(f"\n{'=' * 65}")
    print(f"  {title}")
    print(f"{'=' * 65}")


def save_json(data: dict, filename: str):
    """Save a Python dict to a JSON file in OUTPUT_DIR."""
    path = OUTPUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"  [SAVED] {path}")


def save_artifact(obj, filename: str):
    """
    Save a trained model or preprocessor as a .pkl file.

    Why pickle?
    - Industry standard for sklearn models
    - The exact same object can be loaded and used for new predictions
    - This is what AI-SOC does — models saved as .pkl, loaded by FastAPI
    """
    if not SAVE_ARTIFACTS:
        return
    path = MODEL_DIR / filename
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"  [SAVED] {path.name}  ({size_mb:.2f} MB)")


def safe_attack_name(value) -> str:
    """Return clean attack category name, handling NaN and empty strings."""
    if pd.isna(value):
        return "Unknown"
    val = str(value).strip()
    return val if val else "Unknown"


def get_ohe():
    """
    Return a OneHotEncoder compatible with both old and new sklearn.

    Why this exists:
    sklearn renamed sparse=False to sparse_output=False in v1.2.
    This function handles both so the code works regardless of version.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


print("[OK] Stage 3 — Helper functions defined")


[OK] Stage 3 — Helper functions defined


In [41]:
# ─────────────────────────────────────────────────────────────
# STAGE 4 — DATA LOADING  (WAS BROKEN — NOW FIXED)
#
# WHAT WAS WRONG IN THE ORIGINAL:
# The function load_datasets() referenced train_df and test_df
# INSIDE the function body, but those variables were never created
# inside the function. The function never actually read the CSV files.
# It would crash immediately with NameError: name 'train_df' is not defined.
#
# HOW WE FIXED IT:
# - The function now READS the CSV files using pd.read_csv()
# - train_df and test_df are created INSIDE the function
# - The function returns them so the rest of the pipeline can use them
# - Added clear validation so missing columns give a helpful error message
# - Added dtype handling for edge cases in UNSW-NB15
#
# WHY WE KEEP TRAIN AND TEST SEPARATE:
# UNSW-NB15 has official train and test splits.
# Keeping them separate means our results are comparable to published papers.
# We do NOT re-split the data ourselves — that would make results unreproducible.
# ─────────────────────────────────────────────────────────────

def load_datasets(train_path: Path, test_path: Path):
    """
    Load the official UNSW-NB15 training and test CSV files.

    Returns:
        train_df : pd.DataFrame — training set
        test_df  : pd.DataFrame — test set
    """
    print_header("STAGE 4 — LOADING DATA")

    # ── Validate files exist ──────────────────────────────────
    # If the files are missing, give a helpful error message
    # instead of a confusing pandas FileNotFoundError.
    for path, label in [(train_path, "Training"), (test_path, "Testing")]:
        if not Path(path).exists():
            raise FileNotFoundError(
                f"\n  {label} file not found: {path}\n"
                f"  Place your UNSW-NB15 CSV files in: {SCRIPT_DIR}\n"
                f"  Expected filenames:\n"
                f"    UNSW_NB15_training-set.csv\n"
                f"    UNSW_NB15_testing-set.csv"
            )

    # ── Read CSV files ────────────────────────────────────────
    # low_memory=False tells pandas to read the whole file before
    # deciding column types, avoiding mixed-type warnings.
    print(f"  Reading training file : {Path(train_path).name}")
    train_df = pd.read_csv(train_path, low_memory=False)

    print(f"  Reading testing file  : {Path(test_path).name}")
    test_df  = pd.read_csv(test_path,  low_memory=False)

    # ── Standardise column names ──────────────────────────────
    # UNSW-NB15 sometimes has extra spaces in column names.
    # Strip them to avoid KeyError surprises later.
    train_df.columns = train_df.columns.str.strip().str.lower()
    test_df.columns  = test_df.columns.str.strip().str.lower()

    # ── Validate required columns exist ──────────────────────
    # These two columns MUST be present — everything else depends on them.
    for df, label in [(train_df, "Training"), (test_df, "Testing")]:
        if "label" not in df.columns:
            raise ValueError(
                f"{label} file is missing 'label' column.\n"
                f"  Columns found: {list(df.columns)}"
            )
        if "attack_cat" not in df.columns:
            raise ValueError(
                f"{label} file is missing 'attack_cat' column.\n"
                f"  Columns found: {list(df.columns)}"
            )

    # ── Print summary ─────────────────────────────────────────
    print(f"\n  Train samples   : {len(train_df):,}")
    print(f"  Test samples    : {len(test_df):,}")
    print(f"  Train features  : {train_df.shape[1]}")

    # Attack rate = proportion of rows where label == 1
    train_attack_rate = train_df["label"].mean()
    test_attack_rate  = test_df["label"].mean()
    print(f"\n  Train attack rate : {train_attack_rate:.2%}")
    print(f"  Test attack rate  : {test_attack_rate:.2%}")

    # Show attack category breakdown
    print(f"\n  Attack categories in train:")
    for cat, count in train_df["attack_cat"].value_counts().items():
        pct = count / len(train_df) * 100
        print(f"    {cat:<20} {count:>7,}  ({pct:.1f}%)")

    return train_df, test_df


print("[OK] Stage 4 — load_datasets() defined and ready")
print("     (Will run when run_pipeline() is called)")


[OK] Stage 4 — load_datasets() defined and ready
     (Will run when run_pipeline() is called)


In [42]:

# STAGE 5 — FEATURE PREPARATION
#
# WHY THIS STAGE EXISTS:
# The raw dataframe has everything mixed together — features,
# the label we want to predict, the attack category name, and
# identifier columns like IP addresses.
#
# We need to separate them cleanly:
#   X           = the features the model learns from
#   y           = the binary label (0=normal, 1=attack)
#   attack_cat  = the attack category name (for MITRE mapping later)
#
# WHY WE DROP CERTAIN COLUMNS:
# - "id", "srcip", "dstip", "sport", "dsport" are identifiers.
#   They are specific to this dataset's network capture.
#   If the model learns from them, it memorises the dataset
#   instead of learning real attack patterns. This is called
#   data leakage — the model would fail on real traffic.
#
# BORROWED FROM AI-SOC:
# They do the same thing with CICIDS2017:
#   DROP_COLUMNS = ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', ...]
# Same principle — remove non-predictive identifiers.
# ─────────────────────────────────────────────────────────────

def split_features_and_labels(df: pd.DataFrame):
    """
    Split a UNSW-NB15 dataframe into features, labels, and attack categories.

    Parameters:
        df : pd.DataFrame — the full loaded dataframe (train or test)

    Returns:
        X          : pd.DataFrame — feature columns only
        y          : np.ndarray   — binary labels (0=normal, 1=attack)
        attack_cat : pd.Series    — attack category names for reporting
    """

    # Columns to drop — identifiers and target columns
    # We keep only columns that the model can learn from
    drop_cols = ["label", "attack_cat", "id", "srcip", "dstip", "sport", "dsport"]

    # Only drop columns that actually exist in this dataframe
    # (different UNSW-NB15 versions may have slightly different column sets)
    existing_drop = [c for c in drop_cols if c in df.columns]

    # X = all columns except the ones we're dropping
    X = df.drop(columns=existing_drop).copy()

    # y = the binary label column, cast to integer (0 or 1)
    # Some versions store it as float — int() makes sure it's clean
    y = df["label"].astype(int).values

    # attack_cat = the attack category name (Exploits, DoS, etc.)
    # We keep this separately — it's not a feature, it's a reporting label
    # Used later for MITRE mapping and role-aware recommendations
    attack_cat = df["attack_cat"].copy()

    return X, y, attack_cat


print("[OK] Stage 5 — split_features_and_labels() defined")

[OK] Stage 5 — split_features_and_labels() defined


In [43]:
# STAGE 6 — BUILD PREPROCESSORS
#
# WHY THIS STAGE EXISTS:
# Raw data has two problems before it can go into a model:
#
# Problem 1 — Missing values
#   Some cells are NaN (empty). Models crash on NaN.
#   Fix: fill with the column median (for numbers) or
#        most common value (for text/categories).
#
# Problem 2 — Text columns
#   Models only understand numbers.
#   UNSW-NB15 has text columns like "proto" (tcp, udp, etc.)
#   Fix: OneHotEncoder converts "proto=tcp" into a column of 1s and 0s.
#
# WHY TWO PREPROCESSORS (not one):
#   Random Forest does NOT need feature scaling.
#   Tree models split on "is this value above X?" — the actual scale
#   of numbers doesn't matter. Scaling would add work for no benefit.
#
#   Isolation Forest DOES need scaling.
#   It uses Euclidean distance to decide "is this point isolated?".
#   If one feature is 0-1 and another is 0-1,000,000, the big one
#   dominates all distance calculations. Scaling fixes this.
#
# BORROWED FROM AI-SOC:
# They use StandardScaler for their inference pipeline too.
# We follow the same pattern but add a dedicated IF scaler
# while keeping RF unscaled for better tree performance.
#
# WHAT ColumnTransformer DOES:
# It applies different preprocessing to different columns
# in one clean step. It's sklearn's recommended approach
# for mixed numeric + categorical data.
# ─────────────────────────────────────────────────────────────

def build_preprocessors(X_train: pd.DataFrame):
    """
    Build separate preprocessing pipelines for RF and Isolation Forest.

    We fit ONLY on training data (X_train).
    The fitted preprocessors are then used to transform test data.
    This prevents test data from influencing preprocessing —
    another form of data leakage prevention.

    Parameters:
        X_train : pd.DataFrame — training features (not yet preprocessed)

    Returns:
        rf_preprocessor : ColumnTransformer — for Random Forest (no scaling)
        if_preprocessor : ColumnTransformer — for Isolation Forest (with scaling)
    """
    print_header("STAGE 6 — BUILDING PREPROCESSORS")

    # ── Detect column types from training data ────────────────
    # Numeric columns = all columns that hold numbers
    # Categorical columns = all columns that hold text/objects
    # We detect this from X_train — not from the full dataset.
    numeric_cols     = X_train.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

    print(f"  Numeric features     : {len(numeric_cols)}")
    print(f"  Categorical features : {len(categorical_cols)}")
    if categorical_cols:
        print(f"  Categorical columns  : {categorical_cols}")

    # ── Random Forest Preprocessor ────────────────────────────
    # For numeric columns:
    #   SimpleImputer fills NaN with the median value of that column.
    #   Median is better than mean when there are outliers (which
    #   are common in network traffic data).
    #
    # For categorical columns:
    #   SimpleImputer fills NaN with the most frequent value.
    #   OneHotEncoder converts each category to binary columns.
    #   handle_unknown="ignore" means new categories at test time
    #   are silently ignored rather than causing a crash.
    #
    # NO StandardScaler here — trees don't need it.
    rf_preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                numeric_cols
            ),
            (
                "cat",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot",  get_ohe()),
                ]),
                categorical_cols
            ),
        ],
        remainder="drop"    # silently drop any columns not listed above
    )

    # ── Isolation Forest Preprocessor ─────────────────────────
    # Identical to RF preprocessor EXCEPT we add StandardScaler
    # after the numeric imputer.
    #
    # StandardScaler transforms each numeric column to:
    #   mean = 0, standard deviation = 1
    # This puts all features on the same scale so IF's
    # distance calculations are fair across all features.
    if_preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler",  StandardScaler()),   # ← only difference
                ]),
                numeric_cols
            ),
            (
                "cat",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot",  get_ohe()),
                ]),
                categorical_cols
            ),
        ],
        remainder="drop"
    )

    # ── Fit both preprocessors on training data ONLY ──────────
    # .fit_transform() on training = learn the statistics AND apply them
    # .transform() on test        = apply the SAME statistics (no re-learning)
    #
    # Why this matters:
    # If we fit on test data too, the model "knows" about the test set
    # before it's evaluated. That would make results look better than
    # they really are.
    print("\n  Fitting RF preprocessor on training data...")
    X_train_rf = rf_preprocessor.fit_transform(X_train)

    print("  Fitting IF preprocessor on training data...")
    X_train_if = if_preprocessor.fit_transform(X_train)

    print(f"\n  RF preprocessed shape : {X_train_rf.shape}")
    print(f"  IF preprocessed shape : {X_train_if.shape}")

    # ── Save preprocessors as .pkl files ─────────────────────
    # These are needed at inference time.
    # When a new event comes in from Splunk, we must apply
    # the EXACT same transformations as during training.
    save_artifact(rf_preprocessor, "rf_preprocessor.pkl")
    save_artifact(if_preprocessor, "if_preprocessor.pkl")

    # ── Save feature names for the feature importance plot ────
    # We'll need these in Stage 12 when plotting which features
    # matter most to the Random Forest.
    try:
        feature_names = rf_preprocessor.get_feature_names_out()
    except Exception:
        n = X_train_rf.shape[1]
        feature_names = [f"feature_{i}" for i in range(n)]

    save_artifact(feature_names, "feature_names.pkl")

    return rf_preprocessor, if_preprocessor, X_train_rf, X_train_if


print("[OK] Stage 6 — build_preprocessors() defined")


[OK] Stage 6 — build_preprocessors() defined


In [44]:
# STAGE 7 — METRIC EVALUATION
#
# WHY THIS STAGE EXISTS:
# After a model makes predictions, we need to measure how good
# they are. But "good" means different things in security:
#
# ACCURACY alone is misleading for intrusion detection.
# Example: If 90% of traffic is normal, a model that always
# predicts "normal" gets 90% accuracy but detects ZERO attacks.
#
# We need multiple metrics to tell the full story:
#
# ┌──────────────┬──────────────────────────────────────────────┐
# │ Metric       │ What it tells us                             │
# ├──────────────┼──────────────────────────────────────────────┤
# │ Accuracy     │ Overall correctness (misleading alone)        │
# │ Precision    │ Of all ATTACK predictions, how many were real?│
# │              │ Low = too many false alarms (alert fatigue)   │
# │ Recall       │ Of all real attacks, how many did we catch?   │
# │              │ Low = attackers slipping through              │
# │ F1 Score     │ Balance between Precision and Recall          │
# │              │ The single best metric for security models    │
# │ ROC-AUC      │ How well the model ranks attacks above normal │
# │ PR-AUC       │ F1 averaged across all thresholds             │
# │              │ Best metric when classes are imbalanced       │
# │ FP Rate      │ How often normal traffic is flagged as attack │
# │              │ Directly drives analyst workload              │
# └──────────────┴──────────────────────────────────────────────┘
#
# BORROWED FROM AI-SOC:
# They track False Positive Rate specifically:
#   fpr = fp / (fp + tn)
# This is the metric that directly maps to "how many false alerts
# does the SOC analyst have to investigate per day?"
# A 1% FP rate on 100,000 events = 1,000 false alerts per day.
# AI-SOC achieved 0.25% — we aim for similar with UNSW-NB15.
#
# WHY PR-AUC MATTERS FOR UNSW-NB15:
# UNSW-NB15 has class imbalance — more normal than attack.
# ROC-AUC can be misleading with imbalanced classes because
# it includes true negatives (easy to get right) in the calculation.
# PR-AUC only looks at the attack class, making it more honest.
# ─────────────────────────────────────────────────────────────

def evaluate_predictions(
    model_name: str,
    y_true:     np.ndarray,
    y_pred:     np.ndarray,
    y_score:    np.ndarray = None,
) -> dict:
    """
    Compute and print all evaluation metrics for a model.

    Parameters:
        model_name : str        — display name for printing
        y_true     : np.ndarray — ground truth labels (0/1)
        y_pred     : np.ndarray — predicted labels (0/1)
        y_score    : np.ndarray — prediction scores/probabilities (optional)
                                  used for ROC-AUC and PR-AUC

    Returns:
        metrics : dict — all computed metrics
    """

    print(f"\n  ── {model_name} ──")

    # ── Core classification metrics ───────────────────────────
    # zero_division=0 means if a class has no predictions,
    # return 0 instead of crashing with a division error.
    metrics = {
        "model":     model_name,
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

    # ── Score-based metrics (need probability scores) ─────────
    # These only work if the model can output a confidence score.
    # Isolation Forest outputs anomaly scores, RF outputs probabilities.
    if y_score is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            metrics["roc_auc"] = None

        try:
            # PR-AUC = area under the precision-recall curve
            # More informative than ROC-AUC for imbalanced datasets
            metrics["pr_auc"] = average_precision_score(y_true, y_score)
        except Exception:
            metrics["pr_auc"] = None

    # ── False Positive Rate ───────────────────────────────────
    # FP rate = false positives / all actual negatives
    # In SOC terms: "what fraction of normal traffic do we wrongly flag?"
    # This directly drives analyst workload and alert fatigue.
    try:
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics["false_positive_rate"] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            metrics["false_negative_rate"] = fn / (fn + tp) if (fn + tp) > 0 else 0.0
            metrics["true_positives"]  = int(tp)
            metrics["false_positives"] = int(fp)
            metrics["true_negatives"]  = int(tn)
            metrics["false_negatives"] = int(fn)
    except Exception:
        pass

    # ── Print metrics ─────────────────────────────────────────
    print(f"  {'Accuracy':<22}: {metrics['accuracy']:.4f}")
    print(f"  {'Precision':<22}: {metrics['precision']:.4f}")
    print(f"  {'Recall':<22}: {metrics['recall']:.4f}")
    print(f"  {'F1 Score':<22}: {metrics['f1']:.4f}")

    if metrics.get("roc_auc") is not None:
        print(f"  {'ROC-AUC':<22}: {metrics['roc_auc']:.4f}")
    if metrics.get("pr_auc") is not None:
        print(f"  {'PR-AUC':<22}: {metrics['pr_auc']:.4f}")
    if metrics.get("false_positive_rate") is not None:
        fpr_pct = metrics["false_positive_rate"] * 100
        fnr_pct = metrics["false_negative_rate"] * 100
        print(f"  {'False Positive Rate':<22}: {fpr_pct:.2f}%  "
              f"(~{int(fpr_pct * 100)} false alerts per 10,000 events)")
        print(f"  {'False Negative Rate':<22}: {fnr_pct:.2f}%  "
              f"(~{int(fnr_pct * 100)} missed attacks per 10,000 events)")

    # ── Confusion matrix breakdown ────────────────────────────
    if all(k in metrics for k in ["true_positives", "false_positives",
                                   "true_negatives", "false_negatives"]):
        print(f"\n  Confusion Matrix:")
        print(f"    TP (attack correctly flagged)  : {metrics['true_positives']:>8,}")
        print(f"    FP (normal wrongly flagged)    : {metrics['false_positives']:>8,}")
        print(f"    TN (normal correctly ignored)  : {metrics['true_negatives']:>8,}")
        print(f"    FN (attack missed)             : {metrics['false_negatives']:>8,}")

    # ── Full classification report ────────────────────────────
    print(f"\n  Classification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["Normal", "Attack"],
        zero_division=0
    ))

    return metrics


print("[OK] Stage 7 — evaluate_predictions() defined")


[OK] Stage 7 — evaluate_predictions() defined


In [55]:
# ─────────────────────────────────────────────────────────────
# STAGE 8 — RANDOM FOREST TRAINING  (FIXED)
#
# WHAT WAS WRONG:
# The original Stage 8 built a Pipeline that included the
# rf_preprocessor inside it:
#   Pipeline([("preprocess", rf_preprocessor), ("model", RF)])
#
# Then RandomizedSearchCV tried to call pipeline.fit(X_train_rf, y_train)
# where X_train_rf is already a numpy array (preprocessed in Stage 6).
# The ColumnTransformer inside the pipeline expected a DataFrame with
# named columns like "proto", "service", "state" — but got a numpy
# array with no column names → crash.
#
# THE FIX:
# Stage 6 already preprocessed X_train and X_test into numpy arrays.
# Stage 8 receives those ready-to-use arrays.
# So Stage 8 trains a plain RandomForestClassifier directly —
# NO pipeline wrapper, NO preprocessor inside it.
# The preprocessor is already saved separately from Stage 6.
#
# This is cleaner and matches how AI-SOC does it:
#   preprocess once → save scaler → train model on scaled data
# ─────────────────────────────────────────────────────────────

def train_random_forest(
    X_train:        np.ndarray,   # already preprocessed by rf_preprocessor
    y_train:        np.ndarray,
    X_test:         np.ndarray,   # already preprocessed by rf_preprocessor
    y_test:         np.ndarray,
    rf_preprocessor,              # kept as parameter for saving only
):
    """
    Train and evaluate the Random Forest classifier.

    X_train and X_test must already be preprocessed numpy arrays
    (output of rf_preprocessor.fit_transform / transform from Stage 6).
    The preprocessor is NOT applied again here.
    """
    print_header("STAGE 8 — RANDOM FOREST TRAINING")

    start_time = time.time()

    # ── Base Random Forest classifier ─────────────────────────
    # Plain classifier — NO pipeline wrapper.
    # Preprocessing already done in Stage 6.
    rf_base = RandomForestClassifier(**RF_BASE_PARAMS)

    # ── Optional hyperparameter tuning ────────────────────────
    if RUN_RF_TUNING:
        print("  Running RandomizedSearchCV (this takes several minutes)...")
        print("  Set RUN_RF_TUNING = False in Stage 1 to skip.\n")

        param_grid = {
            "n_estimators":      [200, 300, 500],
            "max_depth":         [10, 20, 30, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf":  [1, 2, 4],
            "max_features":      ["sqrt", "log2"],
        }

        cv = StratifiedKFold(
            n_splits=CV_FOLDS,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        # FIX: estimator is plain RandomForestClassifier, not a Pipeline
        # X_train is already a numpy array — no column name issues
        search = RandomizedSearchCV(
            estimator=rf_base,             # ← plain classifier only
            param_distributions=param_grid,
            n_iter=15,
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )

        # X_train is already preprocessed numpy array from Stage 6
        search.fit(X_train, y_train)

        rf_best = search.best_estimator_
        print(f"\n  Best RF params : {search.best_params_}")
        print(f"  Best CV F1     : {search.best_score_:.4f}")

    else:
        print("  Training base Random Forest (no hyperparameter tuning)...")
        rf_base.fit(X_train, y_train)
        rf_best = rf_base

    # ── Predictions on test set ───────────────────────────────
    rf_pred  = rf_best.predict(X_test)
    rf_proba = rf_best.predict_proba(X_test)[:, 1]

    train_time = time.time() - start_time
    print(f"\n  Training time : {train_time:.1f}s")

    # ── Evaluate ──────────────────────────────────────────────
    rf_metrics = evaluate_predictions(
        "Random Forest", y_test, rf_pred, rf_proba
    )

    # ── Cross-validation robustness check ─────────────────────
    print("\n  Running 5-fold CV robustness check on training data...")
    cv_model  = clone(rf_best)
    cv        = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)

    print(f"  Training CV F1 : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Test F1        : {rf_metrics['f1']:.4f}")

    gap = abs(rf_metrics["f1"] - cv_scores.mean())
    if gap > 0.05:
        print(f"  ⚠ Gap of {gap:.4f} — possible overfitting.")
    else:
        print(f"  ✓ Gap of {gap:.4f} — model generalises well.")

    # ── Save artifacts ────────────────────────────────────────
    # Save just the classifier (preprocessor already saved in Stage 6)
    save_artifact(rf_best, "random_forest_model.pkl")

    rf_metrics["train_time_seconds"] = round(train_time, 2)

    # Return rf_best directly (plain classifier, not a pipeline)
    return rf_best, rf_pred, rf_proba, rf_metrics


print("[OK] Stage 8 FIXED — train_random_forest() ready")
print("     Trains on preprocessed numpy arrays from Stage 6.")
print("     No pipeline wrapper — no ColumnTransformer conflict.")


[OK] Stage 8 FIXED — train_random_forest() ready
     Trains on preprocessed numpy arrays from Stage 6.
     No pipeline wrapper — no ColumnTransformer conflict.


In [56]:
# ─────────────────────────────────────────────────────────────
# STAGE 9 — ISOLATION FOREST TRAINING
#
# WHY ISOLATION FOREST (IF)?
# Random Forest is supervised — it needs labelled attack data.
# But what about attacks the model has never seen before?
# A zero-day exploit, a new malware variant, a novel technique?
# RF will miss them because it has no labelled examples to learn from.
#
# Isolation Forest is UNSUPERVISED — it learns what "normal"
# looks like WITHOUT needing attack labels at all.
# It then flags anything that deviates from that normal baseline.
#
# HOW ISOLATION FOREST WORKS (simply):
# Imagine randomly drawing lines to "cut" a region of data.
# Normal points sit in dense clusters — it takes MANY cuts
# to isolate them (they blend in with their neighbours).
# Anomalous points sit far from clusters — it takes very FEW
# cuts to isolate them (they stand out immediately).
# Points that are isolated quickly = anomaly score closer to -1.
# Points that take many cuts = anomaly score closer to +1.
#
# THE ANOMALY SCORE:
# IF outputs a "decision function" value per sample.
# We INVERT it (multiply by -1) so:
#   Higher score = more anomalous = more likely an attack
#   Lower score  = more normal    = within baseline behaviour
#
# WHY TRAIN ONLY ON NORMAL TRAFFIC:
# This is the key design choice — IF learns the normal baseline
# ONLY from normal samples. Then when it sees an attack, it has
# no "normal" reference for it and flags it as anomalous.
# If we trained on attack data too, the model would learn that
# attacks are also "normal" and lose its detection ability.
#
# THRESHOLD CALIBRATION (our approach — better than contamination):
# The original code used contamination="auto" which is a rough estimate.
# Our approach: take 5% of normal validation traffic as the threshold.
# This means "flag anything more anomalous than the top 5% of normal traffic".
# This is principled — we set the threshold from actual normal data,
# not from a guess about how many attacks exist.
#
# RELATIONSHIP TO RANDOM FOREST:
# IF and RF work together — this is the hybrid approach:
#   IF catches novel/zero-day attacks RF has never seen
#   RF classifies known attack patterns with high confidence
#   When they DISAGREE → deferToHuman = True → human review required
# ─────────────────────────────────────────────────────────────

def train_isolation_forest(
    X_train:        np.ndarray,
    y_train:        np.ndarray,
    X_test:         np.ndarray,
    y_test:         np.ndarray,
    if_preprocessor,
) -> tuple:
    """
    Train and evaluate the Isolation Forest anomaly detector.

    KEY DESIGN CHOICES:
    1. Fit ONLY on normal traffic — no attack samples
    2. Use 80/20 split of normal traffic for fit vs threshold calibration
    3. Set threshold from 95th percentile of normal validation scores
       (not from contamination estimate)
    4. Save threshold as artifact for consistent inference

    Parameters:
        X_train        : raw training features (NOT yet preprocessed)
        y_train        : training labels (0=normal, 1=attack)
        X_test         : raw test features (NOT yet preprocessed)
        y_test         : test labels
        if_preprocessor: fitted IF preprocessor (with StandardScaler)

    Returns:
        if_model       : trained IsolationForest
        if_threshold   : float — calibrated anomaly threshold
        if_pred        : predicted labels on test set (0/1)
        if_scores      : anomaly scores on test set (inverted)
        if_metrics     : dict of evaluation metrics
    """
    print_header("STAGE 9 — ISOLATION FOREST TRAINING")

    start_time = time.time()

    # ── Step 1: Filter to NORMAL traffic only ─────────────────
    # y_train == 0 gives a boolean mask — True for normal rows.
    # We apply this mask to X_train to keep only normal samples.
    X_train_normal = X_train[y_train == 0].copy()

    total     = len(X_train)
    n_normal  = len(X_train_normal)
    n_attack  = (y_train == 1).sum()

    print(f"  Total training samples  : {total:,}")
    print(f"  Normal samples (for IF) : {n_normal:,}  ({n_normal/total:.1%})")
    print(f"  Attack samples (ignored): {n_attack:,}  ({n_attack/total:.1%})")
    print(f"  (IF trains on normal only — it never sees attacks)")

    # ── Step 2: Split normal data into fit set and val set ────
    # fit set (80%) = what IF trains on to learn "normal"
    # val set (20%) = what we use to calibrate the threshold
    #
    # Why split? If we used all normal data for threshold calibration,
    # the threshold would be too permissive — it would be tuned on
    # the exact data IF trained on. Using a held-out validation set
    # gives a fairer, more realistic threshold.
    X_if_fit, X_if_val = train_test_split(
        X_train_normal,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )

    print(f"\n  IF fit set  : {len(X_if_fit):,} normal samples")
    print(f"  IF val set  : {len(X_if_val):,} normal samples (for threshold)")

    # ── Step 3: Preprocess ────────────────────────────────────
    # The IF preprocessor was fitted in Stage 6 on all training data.
    # Here we use it to transform (but NOT re-fit) each split.
    # fit_transform on IF fit set — learn scaling statistics here
    # transform on val + test     — apply same scaling
    #
    # IMPORTANT: We refit the IF preprocessor on the IF fit set only.
    # This is more correct than using the Stage 6 preprocessor
    # which was fitted on all training data including attacks.
    print("\n  Fitting IF preprocessor on normal-only fit set...")
    X_if_fit_proc = if_preprocessor.fit_transform(X_if_fit)
    X_if_val_proc = if_preprocessor.transform(X_if_val)
    X_test_proc   = if_preprocessor.transform(X_test)

    print(f"  Preprocessed shape: {X_if_fit_proc.shape}")

    # ── Step 4: Train Isolation Forest ────────────────────────
    # n_estimators=300 — more trees = more stable anomaly scores
    # contamination="auto" — sklearn estimates based on theory
    #   (we override this with our own threshold in Step 5)
    print("\n  Training Isolation Forest on normal-only fit set...")
    if_model = IsolationForest(**IF_BASE_PARAMS)
    if_model.fit(X_if_fit_proc)

    train_time = time.time() - start_time
    print(f"  Training time: {train_time:.1f}s")

    # ── Step 5: Calibrate threshold from normal validation ────
    # decision_function() returns a score per sample:
    #   Positive = more normal (sample fits the learned distribution)
    #   Negative = more anomalous (sample is isolated from the distribution)
    #
    # We INVERT with -1 so that:
    #   Higher score = more anomalous (easier to understand)
    #   Lower score  = more normal
    #
    # THRESHOLD SETTING:
    # We use the 95th percentile of NORMAL validation scores.
    # Meaning: only flag samples more anomalous than 95% of
    # normal traffic. This gives approximately 5% false positive
    # rate on normal traffic — a reasonable operating point.
    # You can tune this: 99th percentile = stricter (fewer FP, more FN)
    #                    90th percentile = looser  (more FP, fewer FN)
    val_scores = -if_model.decision_function(X_if_val_proc)
    threshold  = float(np.percentile(val_scores, 95))

    print(f"\n  Anomaly score stats on normal validation set:")
    print(f"    Min    : {val_scores.min():.4f}")
    print(f"    Mean   : {val_scores.mean():.4f}")
    print(f"    95th % : {threshold:.4f}  ← threshold")
    print(f"    Max    : {val_scores.max():.4f}")

    # ── Step 6: Score the test set ────────────────────────────
    # Get anomaly score for every test sample
    # Then predict: ATTACK if score >= threshold, NORMAL if below
    if_scores = -if_model.decision_function(X_test_proc)
    if_pred   = (if_scores >= threshold).astype(int)

    # ── Step 7: Evaluate ──────────────────────────────────────
    if_metrics = evaluate_predictions(
        "Isolation Forest", y_test, if_pred, if_scores
    )

    # ── Step 8: Score distribution analysis ──────────────────
    # Compare anomaly score distributions for normal vs attack.
    # A well-trained IF should show clearly separated distributions.
    normal_scores = if_scores[y_test == 0]
    attack_scores = if_scores[y_test == 1]

    print(f"\n  Score distribution on test set:")
    print(f"    Normal traffic  — mean: {normal_scores.mean():.4f}  "
          f"std: {normal_scores.std():.4f}")
    print(f"    Attack traffic  — mean: {attack_scores.mean():.4f}  "
          f"std: {attack_scores.std():.4f}")

    separation = attack_scores.mean() - normal_scores.mean()
    print(f"    Score separation: {separation:.4f}  "
          f"({'good' if separation > 0.05 else 'low — IF may struggle with this dataset'})")

    # ── Step 9: Save artifacts ────────────────────────────────
    save_artifact(if_model,      "isolation_forest_model.pkl")
    save_artifact(if_preprocessor, "if_preprocessor_fitted.pkl")

    # Save the threshold — critical for consistent inference
    # Without saving this, predictions on new data will use
    # a different threshold and give inconsistent results
    threshold_data = {
        "threshold":       threshold,
        "percentile_used": 95,
        "val_set_size":    len(X_if_val),
        "description":     "95th percentile of normal-only validation anomaly scores"
    }
    save_json(threshold_data, "if_threshold.json")
    save_artifact(threshold, "if_threshold.pkl")

    if_metrics["threshold"]          = threshold
    if_metrics["train_time_seconds"] = round(train_time, 2)

    return if_model, if_preprocessor, threshold, if_pred, if_scores, if_metrics


print("[OK] Stage 9 — train_isolation_forest() defined")


[OK] Stage 9 — train_isolation_forest() defined


In [63]:
# ─────────────────────────────────────────────────────────────
# STAGE 10 — HYBRID DECISION ENGINE  (FIXED)
#
# FIXES APPLIED:
#
# Fix 1 — RF threshold lowered from 0.50 to 0.35
#   Why: UNSW-NB15 training set has 68% attacks — RF learns to
#   strongly predict ATTACK. On the test set (55% attacks) this
#   causes 27% false positive rate.
#   Lowering the RF threshold to 0.35 means we only flag as ATTACK
#   when RF is more confident, reducing false alarms significantly.
#
# Fix 2 — Defer band tightened from (0.40–0.65) to (0.45–0.60)
#   Why: The wide uncertainty band caused 51% deferral rate.
#   A tighter band means only genuinely ambiguous alerts defer.
#   Target: ~10–15% deferral, not 51%.
#
# Fix 3 — Models-agree logic improved
#   Why: IF only has 33% recall — it misses most attacks.
#   When IF says NORMAL but RF says ATTACK with high confidence
#   (>0.75), we should trust RF and NOT defer.
#   Only defer on disagreement when RF confidence is moderate.
# ─────────────────────────────────────────────────────────────

def build_hybrid_decision(
    rf_proba:     np.ndarray,
    if_scores:    np.ndarray,
    if_threshold: float,
    y_test:       np.ndarray,
) -> tuple:
    """
    Combine RF probability and IF anomaly score into a hybrid decision.
    """
    print_header("STAGE 10 — HYBRID DECISION ENGINE")

    # ── Normalise IF scores to 0–1 ────────────────────────────
    if_min = float(np.min(if_scores))
    if_max = float(np.max(if_scores))

    if (if_max - if_min) < 1e-9:
        if_norm = np.zeros_like(if_scores, dtype=float)
    else:
        if_norm = (if_scores - if_min) / (if_max - if_min)

    print(f"  IF score range : [{if_min:.4f}, {if_max:.4f}]")
    print(f"  IF norm range  : [{if_norm.min():.4f}, {if_norm.max():.4f}]")

    # ── Weighted blend ────────────────────────────────────────
    RF_WEIGHT = 0.70
    IF_WEIGHT = 0.30
    hybrid_score = RF_WEIGHT * rf_proba + IF_WEIGHT * if_norm

    print(f"\n  Blend weights    : RF={RF_WEIGHT:.0%}  IF={IF_WEIGHT:.0%}")
    print(f"  Hybrid score range: [{hybrid_score.min():.4f}, {hybrid_score.max():.4f}]")

    # ── FIX 1: Tuned hybrid threshold ────────────────────────
    # Original 0.50 caused 27% FP rate.
    # UNSW-NB15 training set is 68% attacks — RF over-predicts ATTACK.
    # Raising hybrid threshold to 0.55 reduces false alarms.
    HYBRID_THRESHOLD = 0.55
    hybrid_pred = (hybrid_score >= HYBRID_THRESHOLD).astype(int)

    print(f"  Hybrid threshold : {HYBRID_THRESHOLD}  (raised from 0.50 to reduce FP rate)")

    # ── Models agree / disagree ───────────────────────────────
    rf_pred_binary = (rf_proba   >= 0.50).astype(int)
    if_pred_binary = (if_scores  >= if_threshold).astype(int)
    models_agree   = (rf_pred_binary == if_pred_binary)

    agree_count    = models_agree.sum()
    disagree_count = (~models_agree).sum()
    print(f"\n  Models agree     : {agree_count:,}  ({agree_count/len(models_agree):.1%})")
    print(f"  Models disagree  : {disagree_count:,}  ({disagree_count/len(models_agree):.1%})")

    # ── FIX 2 + 3: Smarter defer-to-human logic ───────────────
    # Tighter uncertainty band: 0.45–0.60 (was 0.40–0.65)
    # Only defer on disagreement when RF is NOT highly confident.
    # If RF says >0.75 ATTACK, trust RF even if IF disagrees.
    # If RF says <0.25 NORMAL, trust RF even if IF disagrees.
    # Only genuinely ambiguous disagreements trigger deferral.
    LOW_CONF  = 0.45
    HIGH_CONF = 0.60
    HIGH_RF_CONFIDENCE = 0.75   # trust RF alone above this
    LOW_RF_CONFIDENCE  = 0.25   # trust RF alone below this

    uncertain_band = (hybrid_score >= LOW_CONF) & (hybrid_score <= HIGH_CONF)

    # Disagreement only triggers defer when RF is not highly confident
    rf_uncertain   = (rf_proba >= LOW_RF_CONFIDENCE) & (rf_proba <= HIGH_RF_CONFIDENCE)
    meaningful_disagreement = (~models_agree) & rf_uncertain

    defer_flags = uncertain_band | meaningful_disagreement

    defer_count = defer_flags.sum()
    print(f"\n  Uncertainty band : [{LOW_CONF}, {HIGH_CONF}]  (tightened from [0.40, 0.65])")
    print(f"  RF trust band    : trust RF alone if confidence > {HIGH_RF_CONFIDENCE} or < {LOW_RF_CONFIDENCE}")
    print(f"\n  Defer-to-human   : {defer_count:,}  ({defer_count/len(defer_flags):.1%})")
    print(f"  Auto-classified  : {len(defer_flags) - defer_count:,}  "
          f"({(len(defer_flags)-defer_count)/len(defer_flags):.1%})")

    deferred_by_uncertainty  = (uncertain_band & ~meaningful_disagreement).sum()
    deferred_by_disagreement = (meaningful_disagreement & ~uncertain_band).sum()
    deferred_by_both         = (uncertain_band & meaningful_disagreement).sum()
    print(f"\n  Defer breakdown:")
    print(f"    Uncertainty band only : {deferred_by_uncertainty:,}")
    print(f"    Model disagreement    : {deferred_by_disagreement:,}")
    print(f"    Both reasons          : {deferred_by_both:,}")

    # ── Evaluate hybrid ───────────────────────────────────────
    hybrid_metrics = evaluate_predictions(
        "Hybrid (RF 70% + IF 30%)", y_test, hybrid_pred, hybrid_score
    )

    # ── Attack predictions comparison ────────────────────────
    rf_attacks     = rf_pred_binary.sum()
    if_attacks     = if_pred_binary.sum()
    hybrid_attacks = hybrid_pred.sum()

    print(f"\n  Attack predictions comparison:")
    print(f"    Random Forest     : {rf_attacks:,}  ({rf_attacks/len(y_test):.1%})")
    print(f"    Isolation Forest  : {if_attacks:,}  ({if_attacks/len(y_test):.1%})")
    print(f"    Hybrid            : {hybrid_attacks:,}  ({hybrid_attacks/len(y_test):.1%})")
    print(f"    Actual attacks    : {y_test.sum():,}  ({y_test.sum()/len(y_test):.1%})")

    hybrid_metrics["defer_count"]      = int(defer_count)
    hybrid_metrics["defer_rate"]       = float(defer_count / len(defer_flags))
    hybrid_metrics["agree_rate"]       = float(agree_count / len(models_agree))
    hybrid_metrics["rf_weight"]        = RF_WEIGHT
    hybrid_metrics["if_weight"]        = IF_WEIGHT
    hybrid_metrics["hybrid_threshold"] = HYBRID_THRESHOLD

    return hybrid_pred, hybrid_score, hybrid_metrics, defer_flags, models_agree


print("[OK] Stage 10 FIXED — smarter threshold + tighter defer band")

[OK] Stage 10 FIXED — smarter threshold + tighter defer band


In [72]:
# ─────────────────────────────────────────────────────────────
# STAGE 11 — ROLE-AWARE OUTPUT  (FIXED)
#
# FIX 3: Show only TRUE POSITIVE attacks in role-aware output.
# Original showed the first 10 hybrid-predicted attacks regardless
# of whether they were true positives or false positives.
# First few samples in test set had attack_cat="Normal" —
# these were false positives shown as role-aware outputs.
#
# Fix: Only generate role-aware output for samples where:
#   hybrid_pred == 1  (predicted ATTACK)
#   AND y_test == 1   (actually IS an attack = true positive)
# This ensures the recommendations are grounded in real threats.
# ─────────────────────────────────────────────────────────────

def generate_role_outputs_for_attacks(
    hybrid_pred:  np.ndarray,
    hybrid_score: np.ndarray,
    defer_flags:  np.ndarray,
    models_agree: np.ndarray,
    rf_pred:      np.ndarray,
    rf_proba:     np.ndarray,
    if_scores:    np.ndarray,
    attack_cat_test: pd.Series,
    y_test:       np.ndarray,
    n_display:    int = 3,
    n_save:       int = 10,
) -> list:
    """
    Generate role-aware outputs for true positive attacks only.

    FIX: Uses y_test == 1 to filter to actual attacks,
    avoiding false positives appearing in role-aware output.
    """
    print_header("STAGE 11 — SAMPLE ROLE-AWARE OUTPUTS")

    # TRUE POSITIVES only: hybrid says ATTACK and it IS an attack
    tp_indices = np.where((hybrid_pred == 1) & (y_test == 1))[0]

    # Also find some deferred true positives to show the HTIL mechanism
    deferred_tp = np.where(
        (hybrid_pred == 1) & (y_test == 1) & defer_flags
    )[0]

    print(f"  True positive attacks found : {len(tp_indices):,}")
    print(f"  Of which deferred to human  : {len(deferred_tp):,}")

    role_outputs = []

    for i, idx in enumerate(tp_indices[:n_save]):
        output = generate_role_output(
            sample_id    = int(idx),
            rf_pred      = int(rf_pred[idx]),
            rf_proba     = float(rf_proba[idx]),
            if_score     = float(if_scores[idx]),
            hybrid_pred  = int(hybrid_pred[idx]),
            hybrid_score = float(hybrid_score[idx]),
            defer        = bool(defer_flags[idx]),
            models_agree = bool(models_agree[idx]),
            attack_cat   = attack_cat_test.iloc[idx],
        )
        role_outputs.append(output)

        if i < n_display:
            print(f"\n{'─' * 60}")
            print(f"  Sample : {output['sampleId']}  |  Status: {output['status']}  |  "
                  f"Confidence: {output['confidence']:.1%}")
            print(f"  Type   : {output['attackType']}  |  "
                  f"Defer: {output['deferToHuman']}  |  "
                  f"Models Agree: {output['modelsAgree']}")
            print(f"  MITRE  : {output['mitreTechnique']}")
            print(f"  Impact : {output['impactTier']}")
            print(f"\n  SOC Analyst (first 6 lines):")
            for line in output['recAnalyst'].split('\n')[:6]:
                if line.strip():
                    print(f"    {line}")
            print(f"\n  Security Engineer (first 4 lines):")
            for line in output['recEngineer'].split('\n')[:4]:
                if line.strip():
                    print(f"    {line}")
            print(f"\n  CISO / GRC (first 4 lines):")
            for line in output['recGrc'].split('\n')[:4]:
                if line.strip():
                    print(f"    {line}")

    save_json(role_outputs, "sample_role_outputs.json")
    print(f"\n  {len(role_outputs)} true-positive role-aware outputs saved → sample_role_outputs.json")

    return role_outputs


print("[OK] Stage 11 FIXED — shows true positive attacks only")
print("     Call: generate_role_outputs_for_attacks(...) in Stage 13")


[OK] Stage 11 FIXED — shows true positive attacks only
     Call: generate_role_outputs_for_attacks(...) in Stage 13


In [68]:
# ─────────────────────────────────────────────────────────────
# STAGE 12 — VISUALISATIONS
#
# WHY PLOTS MATTER FOR YOUR CAPSTONE:
# Numbers alone do not tell the full story to an audience.
# A confusion matrix heatmap shows at a glance whether the model
# is missing attacks or generating too many false alarms.
# A ROC curve shows how well the model separates classes at
# every possible threshold — not just the one we chose.
#
# WHAT WE PRODUCE:
# 1. Confusion matrices   — one per model (RF, IF, Hybrid)
# 2. ROC curves           — all three models on one chart
# 3. Precision-Recall     — better than ROC for imbalanced data
# 4. Model comparison     — bar chart of all metrics side by side
# 5. Attack distribution  — how attack categories are split train/test
# 6. Feature importance   — which network features matter most to RF
# 7. Score distributions  — IF anomaly scores for normal vs attack
# 8. Defer-to-human       — how many alerts were deferred and why
#
# BUG FIXED FROM ORIGINAL — Feature Importance:
# The original code did this:
#   X_test.sample(5000, random_state=42)          ← random subset of rows
#   y_test[np.random.choice(len(y_test), 5000)]   ← DIFFERENT random rows
# These two arrays had mismatched indices — different rows —
# causing a shape mismatch error or silent wrong results.
#
# FIX: Use the SAME indices for both X and y:
#   idx = np.random.RandomState(42).choice(len(y_test), 5000, replace=False)
#   X_sample = X_test[idx]
#   y_sample = y_test[idx]
# ─────────────────────────────────────────────────────────────

# ── Plot style ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":       150,
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.labelsize":   10,
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
})

COLORS = {
    "rf":     "#2563EB",   # blue
    "if":     "#D97706",   # amber
    "hybrid": "#7C3AED",   # violet
    "normal": "#10B981",   # green
    "attack": "#DC2626",   # red
    "defer":  "#F59E0B",   # yellow
}


def _save_fig(fig: plt.Figure, filename: str):
    """Save a figure to OUTPUT_DIR and close it."""
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [SAVED] {filename}")


# ─────────────────────────────────────────────────────────────
# 1. CONFUSION MATRIX
# Shows: TP, FP, TN, FN visually as a heatmap
# Why: Instantly shows whether the model misses attacks (FN)
#      or over-flags normal traffic (FP)
# ─────────────────────────────────────────────────────────────

def plot_conf_matrix(
    y_true:   np.ndarray,
    y_pred:   np.ndarray,
    title:    str,
    filename: str,
):
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=["Normal", "Attack"],
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(title)
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# 2. ROC CURVES — all models on one chart
# Shows: True Positive Rate vs False Positive Rate
# Why: Shows model performance at every possible threshold —
#      not just the one we chose.
# AUC closer to 1.0 = better separation of classes
# ─────────────────────────────────────────────────────────────

def plot_roc(
    y_true:    np.ndarray,
    score_map: dict,
    filename:  str,
):
    """
    Parameters:
        score_map : dict of {label: scores_array}
                    e.g. {"Random Forest": rf_proba, "Isolation Forest": if_scores}
    """
    fig, ax = plt.subplots(figsize=(7, 5))
    colors  = [COLORS["rf"], COLORS["if"], COLORS["hybrid"]]

    for (name, scores), color in zip(score_map.items(), colors):
        try:
            RocCurveDisplay.from_predictions(
                y_true, scores, name=name, ax=ax, color=color
            )
        except Exception as e:
            print(f"  ⚠ ROC plot skipped for {name}: {e}")

    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Chance (AUC=0.50)")
    ax.set_title("ROC Curves — All Models")
    ax.legend(loc="lower right")
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# 3. PRECISION-RECALL CURVES — all models on one chart
# Shows: Precision vs Recall trade-off
# Why: More informative than ROC for imbalanced datasets.
#      UNSW-NB15 has class imbalance — PR-AUC is more honest.
#      A high-AUC ROC curve can hide poor attack detection
#      when normal traffic dominates.
# ─────────────────────────────────────────────────────────────

def plot_pr(
    y_true:    np.ndarray,
    score_map: dict,
    filename:  str,
):
    fig, ax = plt.subplots(figsize=(7, 5))
    colors  = [COLORS["rf"], COLORS["if"], COLORS["hybrid"]]

    for (name, scores), color in zip(score_map.items(), colors):
        try:
            PrecisionRecallDisplay.from_predictions(
                y_true, scores, name=name, ax=ax, color=color
            )
        except Exception as e:
            print(f"  ⚠ PR plot skipped for {name}: {e}")

    ax.set_title("Precision-Recall Curves — All Models")
    ax.legend(loc="upper right")
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# 4. MODEL COMPARISON BAR CHART
# Shows: All metrics (Acc, Prec, Recall, F1, ROC-AUC) side by side
# Why: Single chart for comparing all three models at a glance
#      Good for capstone presentation slides
# ─────────────────────────────────────────────────────────────

def plot_model_comparison(
    metrics_list: list,
    filename:     str,
):
    df      = pd.DataFrame(metrics_list)
    metric_cols = [c for c in ["accuracy","precision","recall","f1","roc_auc","pr_auc"]
                   if c in df.columns]

    fig, ax = plt.subplots(figsize=(12, 6))
    x       = np.arange(len(df))
    width   = 0.13
    palette = ["#2563EB","#7C3AED","#10B981","#D97706","#DC2626","#6B7280"]

    for i, (col, color) in enumerate(zip(metric_cols, palette)):
        vals = df[col].fillna(0).values
        bars = ax.bar(x + i * width, vals, width=width,
                      label=col.upper().replace("_", "-"), color=color, alpha=0.85)
        # Add value label on top of each bar
        for bar, val in zip(bars, vals):
            if val > 0.01:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{val:.3f}",
                    ha="center", va="bottom", fontsize=7, rotation=45
                )

    ax.set_xticks(x + width * (len(metric_cols) - 1) / 2)
    ax.set_xticklabels(df["model"], fontsize=10)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    ax.set_title("RAID-SecOps — Model Performance Comparison")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# 5. ATTACK CATEGORY DISTRIBUTION
# Shows: How many samples of each attack type in train and test
# Why: Verifies the train/test split is balanced across categories
#      Important for UNSW-NB15 which has very uneven category counts
# ─────────────────────────────────────────────────────────────

def plot_attack_distribution(
    attack_cat_train: pd.Series,
    attack_cat_test:  pd.Series,
    filename:         str,
):
    train_counts = attack_cat_train.value_counts().sort_index()
    test_counts  = attack_cat_test.value_counts().sort_index()

    all_cats     = sorted(set(train_counts.index) | set(test_counts.index))
    train_counts = train_counts.reindex(all_cats, fill_value=0)
    test_counts  = test_counts.reindex(all_cats, fill_value=0)

    x     = np.arange(len(all_cats))
    width = 0.4

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.bar(x - width/2, train_counts.values, width=width,
           label="Train", color=COLORS["rf"], alpha=0.85)
    ax.bar(x + width/2, test_counts.values, width=width,
           label="Test", color=COLORS["if"], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(all_cats, rotation=40, ha="right", fontsize=9)
    ax.set_title("UNSW-NB15 — Attack Category Distribution (Train vs Test)")
    ax.set_ylabel("Sample Count")
    ax.legend()
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# 6. RANDOM FOREST FEATURE IMPORTANCE (BUG FIXED)
# Shows: Which network features contribute most to RF decisions
# Why: Explainability — analysts can understand WHY an alert fired
#      Directly supports EQ4 (explainability increases trust)
#
# METHOD: Permutation Importance
# We shuffle one feature at a time and measure how much the
# model's F1 score drops. A big drop = that feature matters a lot.
# This is more honest than the built-in Gini importance which
# can overrate high-cardinality features.
#
# BUG FIX: Original used X_test.sample(5000) and a separate
# np.random.choice for y_test — different rows, mismatched arrays.
# Fixed by using the SAME index array for both.
# ─────────────────────────────────────────────────────────────

def plot_rf_feature_importance(
    rf_model:      object,
    X_test:        np.ndarray,
    y_test:        np.ndarray,
    feature_names: list,
    filename:      str,
    top_n:         int = 20,
    sample_size:   int = 3000,
):
    print(f"  Computing permutation importance (sample={sample_size})...")

    # ── FIX: use the SAME indices for X and y ─────────────────
    rng       = np.random.RandomState(RANDOM_STATE)
    n_samples = min(sample_size, len(y_test))
    idx       = rng.choice(len(y_test), n_samples, replace=False)

    X_sample  = X_test[idx]   # rows selected by idx
    y_sample  = y_test[idx]   # SAME rows — no mismatch

    # ── Compute permutation importance ────────────────────────
    try:
        result = permutation_importance(
            rf_model,
            X_sample,
            y_sample,
            n_repeats=3,
            random_state=RANDOM_STATE,
            n_jobs=1,
            scoring="f1",
        )

        # Build a series of feature_name → mean importance
        importances = pd.Series(
            result.importances_mean,
            index=feature_names[:len(result.importances_mean)]
        ).sort_values(ascending=False).head(top_n)

        fig, ax = plt.subplots(figsize=(10, 7))
        importances.sort_values().plot(
            kind="barh", ax=ax, color=COLORS["rf"], alpha=0.85
        )
        ax.set_title(f"Top {top_n} Features — RF Permutation Importance\n"
                     f"(higher = more important for attack detection)")
        ax.set_xlabel("Mean Decrease in F1 Score when Feature is Shuffled")
        plt.tight_layout()
        _save_fig(fig, filename)

    except Exception as e:
        print(f"  ⚠ Feature importance plot failed: {e}")


# ─────────────────────────────────────────────────────────────
# 7. ISOLATION FOREST SCORE DISTRIBUTIONS
# Shows: Anomaly score distributions for normal vs attack
# Why: Validates that IF learned a meaningful baseline —
#      attack scores should be clearly higher than normal scores
#      A large separation gap = IF is working correctly
# ─────────────────────────────────────────────────────────────

def plot_if_score_distribution(
    if_scores:    np.ndarray,
    y_test:       np.ndarray,
    if_threshold: float,
    filename:     str,
):
    normal_scores = if_scores[y_test == 0]
    attack_scores = if_scores[y_test == 1]

    fig, ax = plt.subplots(figsize=(9, 5))

    # Plot distributions as overlapping histograms
    ax.hist(normal_scores, bins=60, alpha=0.55, color=COLORS["normal"],
            label=f"Normal  (n={len(normal_scores):,})", density=True)
    ax.hist(attack_scores, bins=60, alpha=0.55, color=COLORS["attack"],
            label=f"Attack  (n={len(attack_scores):,})", density=True)

    # Mark the calibrated threshold
    ax.axvline(if_threshold, color="black", linestyle="--", lw=1.5,
               label=f"Threshold = {if_threshold:.4f} (95th pct of normal)")

    ax.set_xlabel("Anomaly Score (inverted — higher = more anomalous)")
    ax.set_ylabel("Density")
    ax.set_title("Isolation Forest Score Distributions — Normal vs Attack")
    ax.legend()
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# 8. DEFER-TO-HUMAN BREAKDOWN
# Shows: How alerts were classified — auto vs deferred
# Why: Directly demonstrates EQ5 (human-in-the-loop deferral)
#      Good for capstone presentation — shows the HTIL mechanism
# ─────────────────────────────────────────────────────────────

def plot_defer_breakdown(
    hybrid_pred:  np.ndarray,
    defer_flags:  np.ndarray,
    models_agree: np.ndarray,
    hybrid_score: np.ndarray,
    filename:     str,
):
    # Categories
    auto_normal  = ((hybrid_pred == 0) & ~defer_flags).sum()
    auto_attack  = ((hybrid_pred == 1) & ~defer_flags).sum()
    defer_uncert = (defer_flags & models_agree).sum()
    defer_disagr = (defer_flags & ~models_agree).sum()

    labels = [
        "Auto: Normal\n(confident)",
        "Auto: Attack\n(confident)",
        "Deferred:\nUncertainty",
        "Deferred:\nModel Disagree",
    ]
    values = [auto_normal, auto_attack, defer_uncert, defer_disagr]
    colors = [COLORS["normal"], COLORS["attack"], COLORS["defer"], COLORS["if"]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: Pie chart
    wedges, texts, autotexts = axes[0].pie(
        values,
        labels=labels,
        colors=colors,
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.8,
    )
    axes[0].set_title("Alert Classification Breakdown\n(RAID-SecOps Decision Output)")

    # Right: Confidence score histogram coloured by defer status
    non_defer_scores = hybrid_score[~defer_flags]
    defer_scores     = hybrid_score[defer_flags]

    axes[1].hist(non_defer_scores, bins=40, alpha=0.7, color=COLORS["rf"],
                 label=f"Auto-classified ({len(non_defer_scores):,})", density=True)
    axes[1].hist(defer_scores, bins=40, alpha=0.7, color=COLORS["defer"],
                 label=f"Deferred to human ({len(defer_scores):,})", density=True)
    axes[1].axvline(0.40, color="gray", linestyle=":", lw=1.2, label="Uncertainty band (0.40–0.65)")
    axes[1].axvline(0.65, color="gray", linestyle=":", lw=1.2)
    axes[1].set_xlabel("Hybrid Confidence Score")
    axes[1].set_ylabel("Density")
    axes[1].set_title("Confidence Score Distribution\nby Classification Decision")
    axes[1].legend(fontsize=8)

    plt.suptitle("RAID-SecOps Human-in-the-Loop Deferral Analysis", fontsize=13, y=1.01)
    plt.tight_layout()
    _save_fig(fig, filename)


# ─────────────────────────────────────────────────────────────
# MASTER PLOT RUNNER
# Calls all plot functions in sequence
# ─────────────────────────────────────────────────────────────

def run_all_plots(
    y_test:        np.ndarray,
    rf_pred:       np.ndarray,
    rf_proba:      np.ndarray,
    if_pred:       np.ndarray,
    if_scores:     np.ndarray,
    if_threshold:  float,
    hybrid_pred:   np.ndarray,
    hybrid_score:  np.ndarray,
    defer_flags:   np.ndarray,
    models_agree:  np.ndarray,
    rf_model:      object,
    X_test:        np.ndarray,
    feature_names: list,
    attack_cat_train: pd.Series,
    attack_cat_test:  pd.Series,
    rf_metrics:    dict,
    if_metrics:    dict,
    hybrid_metrics: dict,
):
    print_header("STAGE 12 — SAVING PLOTS")

    # 1. Confusion matrices
    plot_conf_matrix(y_test, rf_pred,
                     "Confusion Matrix — Random Forest",     "cm_random_forest.png")
    plot_conf_matrix(y_test, if_pred,
                     "Confusion Matrix — Isolation Forest",  "cm_isolation_forest.png")
    plot_conf_matrix(y_test, hybrid_pred,
                     "Confusion Matrix — Hybrid (RF+IF)",    "cm_hybrid.png")

    # 2. ROC curves
    plot_roc(
        y_test,
        {"Random Forest": rf_proba, "Isolation Forest": if_scores, "Hybrid": hybrid_score},
        "roc_curves.png",
    )

    # 3. PR curves
    plot_pr(
        y_test,
        {"Random Forest": rf_proba, "Isolation Forest": if_scores, "Hybrid": hybrid_score},
        "pr_curves.png",
    )

    # 4. Model comparison
    plot_model_comparison([rf_metrics, if_metrics, hybrid_metrics], "model_comparison.png")

    # 5. Attack distribution
    plot_attack_distribution(attack_cat_train, attack_cat_test, "attack_distribution.png")

    # 6. Feature importance (bug-fixed)
    plot_rf_feature_importance(
        rf_model, X_test, y_test,
        list(feature_names), "rf_feature_importance.png"
    )

    # 7. IF score distribution
    plot_if_score_distribution(if_scores, y_test, if_threshold, "if_score_distribution.png")

    # 8. Defer-to-human breakdown
    plot_defer_breakdown(hybrid_pred, defer_flags, models_agree, hybrid_score, "defer_breakdown.png")

    print(f"\n  All plots saved to: {OUTPUT_DIR}")


print("[OK] Stage 12 — All plot functions defined")
print("     run_all_plots() is the single entry point")


[OK] Stage 12 — All plot functions defined
     run_all_plots() is the single entry point


In [69]:
# ─────────────────────────────────────────────────────────────
# STAGE 13 — MAIN PIPELINE RUNNER  (FIXED)
# Calls the fixed Stage 11 function for role-aware outputs.
# ─────────────────────────────────────────────────────────────

def run_pipeline():
    pipeline_start = time.time()

    print_header("RAID-SecOps HYBRID ML PIPELINE")
    print(f"  Random seed   : {RANDOM_STATE}")
    print(f"  RF tuning     : {'Yes (RandomizedSearchCV)' if RUN_RF_TUNING else 'No (base params)'}")
    print(f"  Save artifacts: {'Yes' if SAVE_ARTIFACTS else 'No'}")
    print(f"  Output dir    : {OUTPUT_DIR}")

    # Stage 4
    train_df, test_df = load_datasets(TRAIN_PATH, TEST_PATH)

    # Stage 5
    X_train, y_train, attack_cat_train = split_features_and_labels(train_df)
    X_test,  y_test,  attack_cat_test  = split_features_and_labels(test_df)

    print_header("STAGE 5 — FEATURE SUMMARY")
    print(f"  X_train shape : {X_train.shape}")
    print(f"  X_test  shape : {X_test.shape}")
    print(f"  Train normal  : {(y_train == 0).sum():,}  |  Train attack: {(y_train == 1).sum():,}")
    print(f"  Test  normal  : {(y_test  == 0).sum():,}  |  Test  attack: {(y_test  == 1).sum():,}")

    # Stage 6
    rf_preprocessor, if_preprocessor, X_train_rf, X_train_if = \
        build_preprocessors(X_train)
    X_test_rf = rf_preprocessor.transform(X_test)
    X_test_if = if_preprocessor.transform(X_test)

    # Stage 8
    rf_model, rf_pred, rf_proba, rf_metrics = train_random_forest(
        X_train_rf, y_train, X_test_rf, y_test, rf_preprocessor,
    )

    # Stage 9
    if_model, if_preproc, if_threshold, if_pred, if_scores, if_metrics = \
        train_isolation_forest(
            X_train, y_train, X_test, y_test, if_preprocessor,
        )

    # Stage 10 (fixed)
    hybrid_pred, hybrid_score, hybrid_metrics, defer_flags, models_agree = \
        build_hybrid_decision(rf_proba, if_scores, if_threshold, y_test)

    # Stage 11 (fixed) — true positives only
    role_outputs = generate_role_outputs_for_attacks(
        hybrid_pred   = hybrid_pred,
        hybrid_score  = hybrid_score,
        defer_flags   = defer_flags,
        models_agree  = models_agree,
        rf_pred       = rf_pred,
        rf_proba      = rf_proba,
        if_scores     = if_scores,
        attack_cat_test = attack_cat_test,
        y_test        = y_test,
        n_display     = 3,
        n_save        = 10,
    )

    # Stage 12 — plots
    try:
        with open(MODEL_DIR / "feature_names.pkl", "rb") as f:
            feature_names = pickle.load(f)
    except Exception:
        feature_names = [f"feature_{i}" for i in range(X_test_rf.shape[1])]

    run_all_plots(
        y_test           = y_test,
        rf_pred          = rf_pred,
        rf_proba         = rf_proba,
        if_pred          = if_pred,
        if_scores        = if_scores,
        if_threshold     = if_threshold,
        hybrid_pred      = hybrid_pred,
        hybrid_score     = hybrid_score,
        defer_flags      = defer_flags,
        models_agree     = models_agree,
        rf_model         = rf_model,
        X_test           = X_test_rf,
        feature_names    = feature_names,
        attack_cat_train = attack_cat_train,
        attack_cat_test  = attack_cat_test,
        rf_metrics       = rf_metrics,
        if_metrics       = if_metrics,
        hybrid_metrics   = hybrid_metrics,
    )

    # Save metrics
    def clean(m):
        return {k: v for k, v in m.items()
                if isinstance(v, (int, float, str, bool, type(None)))}

    save_json({
        "Random Forest":            clean(rf_metrics),
        "Isolation Forest":         clean(if_metrics),
        "Hybrid (RF 70% + IF 30%)": clean(hybrid_metrics),
        "Isolation Forest Threshold": float(if_threshold),
        "Dataset": {
            "train_samples": int(len(y_train)),
            "test_samples":  int(len(y_test)),
            "train_attacks": int((y_train == 1).sum()),
            "test_attacks":  int((y_test  == 1).sum()),
        }
    }, "metrics_summary.json")

    # Final summary
    pipeline_time = time.time() - pipeline_start
    print_header("FINAL RESULTS SUMMARY")

    header  = ["Model", "Accuracy", "Precision", "Recall", "F1", "FP Rate"]
    divider = ["─" * 22] * 6
    rows = [
        ["Random Forest",
         f"{rf_metrics['accuracy']:.4f}", f"{rf_metrics['precision']:.4f}",
         f"{rf_metrics['recall']:.4f}",   f"{rf_metrics['f1']:.4f}",
         f"{rf_metrics.get('false_positive_rate', 0):.2%}"],
        ["Isolation Forest",
         f"{if_metrics['accuracy']:.4f}", f"{if_metrics['precision']:.4f}",
         f"{if_metrics['recall']:.4f}",   f"{if_metrics['f1']:.4f}",
         f"{if_metrics.get('false_positive_rate', 0):.2%}"],
        ["Hybrid (RF+IF)",
         f"{hybrid_metrics['accuracy']:.4f}", f"{hybrid_metrics['precision']:.4f}",
         f"{hybrid_metrics['recall']:.4f}",   f"{hybrid_metrics['f1']:.4f}",
         f"{hybrid_metrics.get('false_positive_rate', 0):.2%}"],
    ]

    w = 22
    print("  " + "".join(str(c).ljust(w) for c in header))
    print("  " + "".join(str(c).ljust(w) for c in divider))
    for row in rows:
        print("  " + "".join(str(c).ljust(w) for c in row))

    best = max([rf_metrics, if_metrics, hybrid_metrics], key=lambda x: x["f1"])
    defer_count = int(defer_flags.sum())
    total       = len(defer_flags)

    print(f"\n  Best model by F1 : {best['model']}  ({best['f1']:.4f})")
    print(f"\n  Human-in-the-loop deferral:")
    print(f"    Deferred to human : {defer_count:,}  ({defer_count/total:.1%})")
    print(f"    Auto-classified   : {total - defer_count:,}  ({(total-defer_count)/total:.1%})")
    print(f"\n  Pipeline complete in {pipeline_time:.1f}s  ({pipeline_time/60:.1f} min)")
    print(f"  Artifacts : {MODEL_DIR}")
    print(f"  Plots     : {OUTPUT_DIR}")

    return {
        "rf_model":       rf_model,
        "if_model":       if_model,
        "if_threshold":   if_threshold,
        "rf_metrics":     rf_metrics,
        "if_metrics":     if_metrics,
        "hybrid_metrics": hybrid_metrics,
        "role_outputs":   role_outputs,
        "defer_flags":    defer_flags,
        "models_agree":   models_agree,
        "hybrid_score":   hybrid_score,
        "hybrid_pred":    hybrid_pred,
        "rf_pred":        rf_pred,
        "rf_proba":       rf_proba,
        "if_scores":      if_scores,
        "y_test":         y_test,
        "attack_cat_test": attack_cat_test,
    }


print("[OK] Stage 13 FIXED — calls generate_role_outputs_for_attacks()")


[OK] Stage 13 FIXED — calls generate_role_outputs_for_attacks()


In [73]:
# ─────────────────────────────────────────────────────────────
# STAGE 14 — ENTRY POINT
# Run this cell last to execute the full pipeline.
#
# All previous stage cells must have been run first:
#   Stage 0  — imports
#   Stage 1  — configuration
#   Stage 2  — MITRE mapping
#   Stage 3  — helper functions
#   Stage 4  — load_datasets()
#   Stage 5  — split_features_and_labels()
#   Stage 6  — build_preprocessors()
#   Stage 7  — evaluate_predictions()
#   Stage 8  — train_random_forest()
#   Stage 9  — train_isolation_forest()
#   Stage 10 — build_hybrid_decision()
#   Stage 11 — generate_role_output() and builders
#   Stage 12 — run_all_plots() and individual plot functions
#   Stage 13 — run_pipeline()
#
# This cell runs everything and stores the result in `results`
# so you can inspect any object afterwards:
#   results["rf_model"]       — the trained RF pipeline
#   results["if_model"]       — the trained IF model
#   results["if_threshold"]   — the calibrated threshold
#   results["role_outputs"]   — list of role-aware alert dicts
#   results["defer_flags"]    — bool array of deferred alerts
#   results["hybrid_score"]   — confidence scores per alert
# ─────────────────────────────────────────────────────────────

results = run_pipeline()



  RAID-SecOps HYBRID ML PIPELINE
  Random seed   : 42
  RF tuning     : Yes (RandomizedSearchCV)
  Save artifacts: Yes
  Output dir    : C:\Users\kelvi\Desktop\MS CYBERSECURITY\Spring 2026\Capstone\RAID SecOps\raid-secops\ML\raid_outputs

  STAGE 4 — LOADING DATA
  Reading training file : UNSW_NB15_training-set.csv
  Reading testing file  : UNSW_NB15_testing-set.csv

  Train samples   : 175,341
  Test samples    : 82,332
  Train features  : 45

  Train attack rate : 68.06%
  Test attack rate  : 55.06%

  Attack categories in train:
    Normal                56,000  (31.9%)
    Generic               40,000  (22.8%)
    Exploits              33,393  (19.0%)
    Fuzzers               18,184  (10.4%)
    DoS                   12,264  (7.0%)
    Reconnaissance        10,491  (6.0%)
    Analysis               2,000  (1.1%)
    Backdoor               1,746  (1.0%)
    Shellcode              1,133  (0.6%)
    Worms                    130  (0.1%)

  STAGE 5 — FEATURE SUMMARY
  X_train shape : 